# Predicting the Sale Price of Bulldozers using Machine Learning
> The goal of the project is to predict the sale price of a particular piece of heavy equiment at auction based on it's usage, equipment type, and configuaration.  The data is sourced from auction result postings and includes information on usage and equipment configurations.


## Modelling

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_log_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

### Building a custom Transformers for the pipeline for Preprocessing and preparing the data for modelling

In [ ]:
# Create a pipeline that extracts year, month, day, DayOfWeek from the saledate column and then drop the saledate column
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extracts date components from saledate and drops the original.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if "saledate" in X.columns:
            X["saledate"] = pd.to_datetime(X["saledate"])
            X["saleYear"] = X["saledate"].dt.year
            X["saleMonth"] = X["saledate"].dt.month
            X["saleDayOfWeek"] = X["saledate"].dt.dayofweek
            X["saleDayOfYear"] = X["saledate"].dt.dayofyear
            X.drop("saledate", axis=1, inplace=True)

        return X

class TabularImputerAndEncoder(BaseEstimator, TransformerMixin):
    """
    Learns training medians and category mappings, then applies them to any raw dataset.
    """
    def __init__(self):
        self.training_medians_ = {}
        self.category_mappings_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Learn numeric medians
        for label, content in X.items():
            if pd.api.types.is_numeric_dtype(content):
                if pd.isnull(content).sum() > 0:
                    self.training_medians_[label] = content.median()

        # Learn categorical mappings
        for label, content in X.items():
            if not pd.api.types.is_numeric_dtype(content):
                content_cat = content.astype("category")
                self.category_mappings_[label] = content_cat.cat.categories
        return self

    def transform(self, X):
        X = X.copy()

        # Process Numeric Columns

In [3]:
# Import the Train dataset
df_train = pd.read_csv("data/bluebook-for-bulldozers/Train.csv", low_memory=False, parse_dates=["saledate"])
df_train

,SalesID,SalePrice,MachineID,ModelID,datasource,auctioneerID,YearMade,MachineHoursCurrentMeter,UsageBand,saledate,...,Undercarriage_Pad_Width,Stick_Length,Thumb,Pattern_Changer,Grouser_Type,Backhoe_Mounting,Blade_Type,Travel_Controls,Differential_Type,Steering_Controls
0,1139246,66000,999089,3157,121,3.0,2004,68.0,Low,2006-11-16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139248,57000,117657,77,121,3.0,1996,4640.0,Low,2004-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
2,1139249,10000,434808,7009,121,3.0,2001,2838.0,High,2004-02-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1139251,38500,1026470,332,121,3.0,2001,3486.0,High,2011-05-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139253,11000,1057373,17311,121,3.0,2007,722.0,Medium,2009-07-23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401120,6333336,10500,1840702,21439,149,1.0,2005,NaN,NaN,2011-11-02,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401121,6333337,11000,1830472,21439,149,1.0,2005,NaN,NaN,2011-11-02,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401122,6333338,11500,1887659,21439,149,1.0,2005,NaN,NaN,2011-11-02,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401123,6333341,9000,1903570,21435,149,2.0,2005,NaN,NaN,2011-10-25,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN


In [5]:
# Separate features and labels
train_features = df_train.drop("SalePrice", axis = 1)
train_labels = df_train["SalePrice"]

In [6]:
train_features.shape, train_labels.shape

((401125, 52), (401125,))